# 🔄 Module 03: Output Parsers

---

## What Are Output Parsers?

LLMs return **raw text**. Output Parsers transform that text into **structured Python data** — lists, dicts, Pydantic models, etc.

```
LLM Response: "The capital of France is Paris."
    ↓  Parser
Python dict: {"city": "Paris", "country": "France"}
```

### Why Use Output Parsers?
- 🔧 Get typed Python objects from LLM responses
- ✅ Validate LLM outputs automatically
- 🔁 Retry on parse failures
- 📊 Enable downstream data processing

---

## Available Output Parsers

| Parser | Output Type | Best For |
|--------|-------------|----------|
| `StrOutputParser` | string | Simple text output |
| `JsonOutputParser` | dict | JSON structured data |
| `PydanticOutputParser` | Pydantic model | Validated structured data |
| `CommaSeparatedListOutputParser` | List[str] | Simple lists |
| `XMLOutputParser` | dict | XML-structured output |
| `MarkdownListOutputParser` | List[str] | Markdown bullet lists |
| `DatetimeOutputParser` | datetime | Date/time values |
| `EnumOutputParser` | Enum value | Constrained choices |

---

In [1]:
import os
from dotenv import load_dotenv
load_dotenv(dotenv_path="../.env")

from langchain_groq import ChatGroq
llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)
print("Setup complete ✅")

Setup complete ✅


## 1️⃣ StrOutputParser — The Simplest Parser

In [2]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

# ============================================================
# Without StrOutputParser — returns AIMessage object
# ============================================================
prompt = ChatPromptTemplate.from_template("Tell me a joke about {topic}")
chain_no_parser = prompt | llm

result_no_parser = chain_no_parser.invoke({"topic": "Python"})
print("Without parser - Type:", type(result_no_parser).__name__)
print("Without parser - Access:", result_no_parser.content[:50], "...")

print()

# ============================================================
# With StrOutputParser — returns plain string
# ============================================================
chain_with_parser = prompt | llm | StrOutputParser()

result_with_parser = chain_with_parser.invoke({"topic": "Python"})
print("With parser - Type:", type(result_with_parser).__name__)
print("With parser - Value:", result_with_parser[:80], "...")

Without parser - Type: AIMessage
Without parser - Access: Why did the Python script go to therapy?

Because  ...

With parser - Type: TextAccessor
With parser - Value: Why did the Python script go to therapy?

Because it had a lot of "indent"-ed is ...


## 2️⃣ CommaSeparatedListOutputParser

In [3]:
from langchain_core.output_parsers import CommaSeparatedListOutputParser
from langchain_core.prompts import PromptTemplate

parser = CommaSeparatedListOutputParser()

# The parser provides format instructions to guide the LLM
print("Format instructions:")
print(parser.get_format_instructions())
print()

Format instructions:
Your response should be a list of comma separated values, eg: `foo, bar, baz` or `foo,bar,baz`



In [4]:
# Include format instructions in the prompt
template = PromptTemplate(
    template="List {n} {category}.\n{format_instructions}",
    input_variables=["n", "category"],
    partial_variables={"format_instructions": parser.get_format_instructions()}
)

chain = template | llm | parser

# Returns a Python list!
result = chain.invoke({"n": 5, "category": "programming languages"})

print("Result type:", type(result).__name__)
print("Result:", result)
print("First item:", result[0])
print("Length:", len(result))

Result type: list
Result: ['Python', 'Java', 'JavaScript', 'C++', 'Ruby']
First item: Python
Length: 5


## 3️⃣ JsonOutputParser — Parse JSON Output

In [5]:
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts import ChatPromptTemplate

# ============================================================
# Basic JSON parsing
# ============================================================
parser = JsonOutputParser()

template = ChatPromptTemplate.from_messages([
    ("system", "Always respond with valid JSON only. No markdown, no explanation."),
    ("human", "Give me information about the programming language {language} with fields: name, year_created, creator, paradigm, use_cases (list of 3)")
])

chain = template | llm | parser

result = chain.invoke({"language": "Python"})

print("Result type:", type(result).__name__)
print("\nParsed JSON:")
import json
print(json.dumps(result, indent=2))

Result type: dict

Parsed JSON:
{
  "name": "Python",
  "year_created": 1991,
  "creator": "Guido van Rossum",
  "paradigm": "Multi-paradigm",
  "use_cases": [
    "Web Development",
    "Data Analysis",
    "Artificial Intelligence"
  ]
}


In [6]:
# Access fields as a regular dict
print("Name:", result["name"])
print("Creator:", result["creator"])
print("Use cases:", result["use_cases"])

Name: Python
Creator: Guido van Rossum
Use cases: ['Web Development', 'Data Analysis', 'Artificial Intelligence']


## 4️⃣ PydanticOutputParser — Type-Safe Structured Output

In [7]:
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field, validator
from typing import List, Optional

# ============================================================
# Step 1: Define your data schema with Pydantic
# ============================================================
class Person(BaseModel):
    name: str = Field(description="Full name of the person")
    age: int = Field(description="Age in years", ge=0, le=150)
    occupation: str = Field(description="Their main job or role")
    skills: List[str] = Field(description="List of 3-5 key skills")
    biography: str = Field(description="A 2-3 sentence biography")

# ============================================================
# Step 2: Create the parser
# ============================================================
parser = PydanticOutputParser(pydantic_object=Person)

print("Format instructions (sent to LLM):")
print(parser.get_format_instructions()[:500], "...")

Format instructions (sent to LLM):
The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"properties": {"name": {"description": "Full nam ...


In [8]:
# ============================================================
# Step 3: Build the chain
# ============================================================
from langchain_core.prompts import ChatPromptTemplate

template = ChatPromptTemplate.from_messages([
    ("system", "You are a creative writer. Generate fictional person profiles.\n{format_instructions}"),
    ("human", "Create a fictional profile for a {role} from {country}.")
])

# Inject format instructions
template = template.partial(format_instructions=parser.get_format_instructions())

chain = template | llm | parser

# Get a fully typed Python object!
person = chain.invoke({"role": "AI researcher", "country": "Japan"})

print("Type:", type(person).__name__)
print(f"\nName: {person.name}")
print(f"Age: {person.age}")
print(f"Occupation: {person.occupation}")
print(f"Skills: {person.skills}")
print(f"\nBiography:\n{person.biography}")

Type: Person

Name: Kenji Nakamura
Age: 42
Occupation: Artificial Intelligence Researcher
Skills: ['Machine Learning', 'Natural Language Processing', 'Computer Vision']

Biography:
Kenji Nakamura is a renowned AI researcher from Tokyo, Japan. He has spent over a decade developing innovative AI solutions for various industries, including healthcare and finance. Currently, he leads a team of researchers at a prestigious university, focusing on the development of more intelligent and human-like AI systems.


In [9]:
# ============================================================
# Complex Nested Pydantic Model
# ============================================================
from pydantic import BaseModel, Field
from typing import List

class Technology(BaseModel):
    name: str
    version: str
    purpose: str

class TechStack(BaseModel):
    project_name: str = Field(description="Name of the project")
    description: str = Field(description="Brief project description")
    frontend: List[Technology] = Field(description="Frontend technologies used")
    backend: List[Technology] = Field(description="Backend technologies used")
    database: Technology = Field(description="Primary database")
    estimated_cost_per_month_usd: float = Field(description="Estimated monthly hosting cost")

parser = PydanticOutputParser(pydantic_object=TechStack)

template = ChatPromptTemplate.from_messages([
    ("system", "You are a solutions architect. Design tech stacks for projects.\n{format_instructions}"),
    ("human", "Design a modern tech stack for {project_type}")
])
template = template.partial(format_instructions=parser.get_format_instructions())

chain = template | llm | parser
stack = chain.invoke({"project_type": "a real-time collaborative code editor"})

print(f"Project: {stack.project_name}")
print(f"Description: {stack.description}")
print(f"\nFrontend:")
for tech in stack.frontend:
    print(f"  - {tech.name} v{tech.version}: {tech.purpose}")
print(f"\nBackend:")
for tech in stack.backend:
    print(f"  - {tech.name} v{tech.version}: {tech.purpose}")
print(f"\nDatabase: {stack.database.name}")
print(f"Monthly Cost: ${stack.estimated_cost_per_month_usd:.2f}")

Project: Real-Time Collaborative Code Editor
Description: A web-based code editor that allows multiple users to collaborate in real-time

Frontend:
  - React v18.2.0: Frontend framework for building user interface components
  - Redux v8.0.2: State management library for managing global state
  - WebSockets v1.0: Real-time communication protocol for collaborative editing
  - Monaco Editor v0.34.0: Code editor library for syntax highlighting and editing features

Backend:
  - Node.js v16.17.0: Server-side runtime environment for handling requests and WebSocket connections
  - Express.js v4.17.1: Web framework for building RESTful APIs and handling requests
  - Socket.IO v4.5.4: Real-time communication library for managing WebSocket connections
  - Redis v7.0.4: In-memory data store for storing collaborative editing data

Database: PostgreSQL
Monthly Cost: $500.00


## 5️⃣ with_structured_output() — The Modern Approach

In [10]:
# ============================================================
# with_structured_output() — Uses function calling under the hood
# This is the RECOMMENDED modern approach!
# ============================================================
from pydantic import BaseModel, Field
from typing import List

class Recipe(BaseModel):
    """A cooking recipe with ingredients and steps."""
    name: str = Field(description="Name of the dish")
    cuisine: str = Field(description="Type of cuisine")
    prep_time_minutes: int
    cook_time_minutes: int
    servings: int
    ingredients: List[str]
    steps: List[str]
    difficulty: str = Field(description="Easy, Medium, or Hard")

# Bind structured output to the model directly
structured_llm = llm.with_structured_output(Recipe)

recipe = structured_llm.invoke("Give me a recipe for chocolate chip cookies")

print(f"🍪 {recipe.name}")
print(f"Cuisine: {recipe.cuisine}")
print(f"Prep: {recipe.prep_time_minutes} min | Cook: {recipe.cook_time_minutes} min")
print(f"Servings: {recipe.servings} | Difficulty: {recipe.difficulty}")
print(f"\nIngredients ({len(recipe.ingredients)}):")
for ing in recipe.ingredients:
    print(f"  • {ing}")
print(f"\nSteps ({len(recipe.steps)}):")
for i, step in enumerate(recipe.steps, 1):
    print(f"  {i}. {step}")

🍪 Chocolate Chip Cookies
Cuisine: American
Prep: 15 min | Cook: 10 min
Servings: 12 | Difficulty: Easy

Ingredients (8):
  • 2 1/4 cups all-purpose flour
  • 1 tsp baking soda
  • 1 tsp salt
  • 1 cup unsalted butter
  • 3/4 cup white granulated sugar
  • 3/4 cup brown sugar
  • 2 large eggs
  • 2 cups semi-sweet chocolate chips

Steps (8):
  1. Preheat oven to 375°F
  2. Whisk together flour, baking soda, and salt
  3. Cream together butter and sugars
  4. Beat in eggs
  5. Stir in flour mixture
  6. Stir in chocolate chips
  7. Drop by spoonfuls onto baking sheet
  8. Bake for 10 minutes


## 6️⃣ OutputFixingParser — Auto-Retry on Parse Errors

In [14]:
import os
from pydantic import BaseModel
from langchain_core.output_parsers import PydanticOutputParser
# FIX 1: Import OutputFixingParser from the correct updated module
from langchain_community.output_parsers import OutputFixingParser
# FIX 2: Import Groq wrapper for your LLM
from langchain_groq import ChatGroq

# FIX 3: Initialize Groq (ensure your GROQ_API_KEY environment variable is set)
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0.0
)

# ============================================================
# OutputFixingParser wraps another parser and retries on failure
# It asks the LLM to FIX its own malformed output
# ============================================================

class StockInfo(BaseModel):
    symbol: str
    company_name: str
    sector: str
    market_cap_billions: float

base_parser = PydanticOutputParser(pydantic_object=StockInfo)

# Wraps base_parser with auto-fixing capability
fixing_parser = OutputFixingParser.from_llm(
    parser=base_parser,
    llm=llm
)

# Simulate a malformed response the LLM might produce
malformed_output = """
Here's the stock info:
symbol: AAPL
company name: Apple Inc.
sector: Technology
market cap: 3000 billion
"""

try:
    # This will fail because the format isn't right
    result = base_parser.parse(malformed_output)
except Exception as e:
    print(f"Base parser failed: {type(e).__name__}")

# OutputFixingParser will auto-fix it!
result = fixing_parser.parse(malformed_output)
print(f"\nFixed output:")
print(f"Symbol: {result.symbol}")
print(f"Company: {result.company_name}")
print(f"Sector: {result.sector}")
print(f"Market Cap: ${result.market_cap_billions}B")

ImportError: cannot import name 'OutputFixingParser' from 'langchain_community.output_parsers' (c:\Users\sujat\projects\AI\.venv\Lib\site-packages\langchain_community\output_parsers\__init__.py)

## ✅ Module 03 Summary

You've learned:
- ✅ `StrOutputParser` — extract plain text
- ✅ `CommaSeparatedListOutputParser` — extract Python lists
- ✅ `JsonOutputParser` — extract dicts from JSON
- ✅ `PydanticOutputParser` — extract fully typed, validated objects
- ✅ `with_structured_output()` — modern function-calling approach (recommended)
- ✅ `OutputFixingParser` — auto-retry on parse failures

### 🚀 Next: [Module 04 — LCEL: LangChain Expression Language](04_LCEL_Chains.ipynb)